# Testing scipy linear programming

## Imports, definitions and base tests

In [1]:
# Module imports
import sys

import numpy as np
import scipy as sp
from loguru import logger



In [2]:
# Set up the logger
logger.remove()
logger.add(
    sink=sys.stdout,
    format="<level>{level:<10} | {message}</>",
    level="INFO",
    colorize=True,
)

1

### Behaviors as vectors and notation

Suppose a (2,2,2) routed Bell experiment.

$ p(ab|xyz) \in \mathbb{R}^{32}$ is the observed behavior, assumed to be no-signaling : $$p \in \mathcal{NS}$$

We wish for easy conversion from a vector format (for algebraic operations) to a matrix format (for leggibility) :
$$p=\begin{pmatrix} p_{00|00S} \\ p_{00|01S} \\ \vdots \\ p_{00|00L} \\ \vdots \end{pmatrix}\quad \leftrightarrow \quad p=\begin{pmatrix} p_{00|00}& p_{00|01}& p_{00|10}& p_{00|11}\\ p_{01|00}& p_{01|01}& \dots\\ \vdots & & \ddots \\ & & & p_{11|11} \end{pmatrix}$$
where, to account for both matrices $p(z=S)$ and $p(z=L)$, both cases are represented in two matrices, making $p$ either a column vector in $\mathbb{R}^{32}$ or a third-order tensor of shape $(2,4,4)$.

We defined in the Behavior class (behavior.py) some utility functions.

In [3]:
from behaviors import Behavior

# The maximally mixed state over the experiment space
I = Behavior((1 / 4) * np.ones(32))  # noqa: E741

# The usual (2,2,2) PR box
SR_pr_box = np.array(
    [ 1/2, 1/2, 1/2, 0, 0, 0, 0, 1/2, 0, 0, 0, 1/2, 1/2, 1/2, 1/2, 0]
)

# The PR box in the experiment space : p(ab|xy) is assumed to be
# independent of the value of z
pr_box = Behavior(np.concatenate((SR_pr_box, SR_pr_box), axis=0))

def to_behavior(arr):
    return Behavior(np.concatenate((arr, arr), axis=0))

non_positive_arr = np.array(
    [-1/2,-1/2,-1/2,0,0,0,0,1/2,0,0,0,1/2,1/2,1/2,1/2,0,],
)
non_normalized_arr = np.array(
    [1/2,1/2,1/2,0,0,0,0,0,0,0,0,1/2,1/2,1/2,1/2,0,]
)
non_ns_arr = np.array(
    [1/2,1/2,0,0,0,0,1/2,1/2,1/2,1/2,0,0,0,0,1/2,1/2,]
)

non_positive = to_behavior(non_positive_arr)
non_normalized = to_behavior(non_normalized_arr)
non_ns = to_behavior(non_ns_arr)


In [4]:
# Sanity check cell, don't mind me

# print("PR box: ", pr_box)
# print("I: ", I)

print("I == I: ", I == I)
print("I == pr_box: ", I == pr_box)
print("I == 0: ", I == 0)
print("I == 0.25: ", I == 0.25)
print("pr_box == pr_box_array: ", pr_box == np.concatenate((SR_pr_box, SR_pr_box), axis=0))

print("I positive: ", I.positivity())
print("I normalized: ", I.normalization())
print("I no-signaling: ", I.no_signaling())

print("PR box positive: ", pr_box.positivity())
print("PR box normalized: ", pr_box.normalization())
print("PR box no-signaling: ", pr_box.no_signaling())

print("Non-positive: ", non_positive.positivity())
print("Non-normalized: ", non_normalized.normalization())
print("Non-no-signaling: ", non_ns.no_signaling())

print("Non-no-signaling is normalized: ", non_ns.is_normalized())

I == I:  True
I == pr_box:  False
I == 0:  False
I == 0.25:  True
pr_box == pr_box_array:  True
I positive:  True
I normalized:  True
I no-signaling:  True
PR box positive:  True
PR box normalized:  True
PR box no-signaling:  True
Non-positive:  False
Non-normalized:  False
WARNING    | Alice no-signaling violated at index [0, 1]: 0.00000 vs 1.00000 (Δ=1.00e+00)
WARNING    | Alice no-signaling violated at index [1, 1]: 0.00000 vs 1.00000 (Δ=1.00e+00)
WARNING    | Alice no-signaling violated at index [2, 1]: 0.00000 vs 1.00000 (Δ=1.00e+00)
WARNING    | Alice no-signaling violated at index [3, 1]: 0.00000 vs 1.00000 (Δ=1.00e+00)
WARNING    | Alice no-signaling violated at index [4, 1]: 1.00000 vs 0.00000 (Δ=1.00e+00)
WARNING    | Alice no-signaling violated at index [5, 1]: 1.00000 vs 0.00000 (Δ=1.00e+00)
WARNING    | Alice no-signaling violated at index [6, 1]: 1.00000 vs 0.00000 (Δ=1.00e+00)
WARNING    | Alice no-signaling violated at index [7, 1]: 1.00000 vs 0.00000 (Δ=1.00e+00)
Non-n

In [5]:
# Checking the indices correspondence functions

from behaviors import routed_index_to_indices, routed_indices_to_index

for i in range(2*2**2*2**2):
    print(f"i={i} -> routed index: {routed_index_to_indices(i, m=2)}, computed i: {routed_indices_to_index(*routed_index_to_indices(i,m=2), m=2)}")  # noqa: E501

i=0 -> routed index: (0, 0, 0, 0, 0), computed i: 0
i=1 -> routed index: (0, 0, 0, 1, 0), computed i: 1
i=2 -> routed index: (0, 0, 1, 0, 0), computed i: 2
i=3 -> routed index: (0, 0, 1, 1, 0), computed i: 3
i=4 -> routed index: (0, 1, 0, 0, 0), computed i: 4
i=5 -> routed index: (0, 1, 0, 1, 0), computed i: 5
i=6 -> routed index: (0, 1, 1, 0, 0), computed i: 6
i=7 -> routed index: (0, 1, 1, 1, 0), computed i: 7
i=8 -> routed index: (1, 0, 0, 0, 0), computed i: 8
i=9 -> routed index: (1, 0, 0, 1, 0), computed i: 9
i=10 -> routed index: (1, 0, 1, 0, 0), computed i: 10
i=11 -> routed index: (1, 0, 1, 1, 0), computed i: 11
i=12 -> routed index: (1, 1, 0, 0, 0), computed i: 12
i=13 -> routed index: (1, 1, 0, 1, 0), computed i: 13
i=14 -> routed index: (1, 1, 1, 0, 0), computed i: 14
i=15 -> routed index: (1, 1, 1, 1, 0), computed i: 15
i=16 -> routed index: (0, 0, 0, 0, 1), computed i: 16
i=17 -> routed index: (0, 0, 0, 1, 1), computed i: 17
i=18 -> routed index: (0, 0, 1, 0, 1), computed 

In [6]:
# Checking that the no-signaling set is well defined in no_signaling_set.py

from no_signaling_set import routed_no_signaling_equations
from behaviors import completely_mixed_behavior, pr_box

delta, m = 2, 2
equations, right_side = routed_no_signaling_equations(delta, m)


# print(f"Delta: {delta}, m: {m}")
# print("\n--------------\n")
# print(f"Equations shape: {equations.shape}")
# print("\n--------------\n")

# for line in equations:
#     print(str(line).strip("[]").replace("\n", "").replace(" ", "").replace("-1", "2").replace(".", "").replace("0", "."))  # noqa: E501

# print("\n--------------\n")
# print(f"Right side shape: {right_side.shape}")
# print("\n--------------\n")
# print(f"Right side: {right_side}")


print(np.all(equations @ completely_mixed_behavior.get_vector() == right_side))
print(np.all(equations @ pr_box.get_vector() == right_side))



True
True


## Sampling behaviors

In [7]:
from samplers import UniformNormalizedSampler

sampler = UniformNormalizedSampler(2,2,True) # Samples normalized behaviors

sample_behavior = sampler.sample()
print("Sampled behavior: ", sample_behavior)
print("Sample behavior is normalized: ", sample_behavior.is_normalized())
print("Sample behavior is no-signaling: ", sample_behavior.no_signaling())

Sampled behavior:  Behavior:
Short path (z=S):
[[0.5844025  0.30148936 0.72779158 0.05839087]
 [0.08265974 0.60345828 0.02387189 0.37508961]
 [0.05807847 0.01963933 0.10137787 0.12042845]
 [0.27485929 0.07541303 0.14695866 0.44609107]]
Long path (z=L) :
[[0.13726301 0.58605464 0.25384639 0.44481036]
 [0.58987219 0.01630285 0.39458019 0.17994111]
 [0.10455868 0.36040724 0.21422997 0.32567248]
 [0.16830611 0.03723526 0.13734345 0.04957606]]
------------
Sample behavior is normalized:  True
WARNING    | Alice no-signaling violated at index [0, 1]: 0.82917 vs 0.64248 (Δ=1.87e-01)
WARNING    | Alice no-signaling violated at index [1, 1]: 0.46808 vs 0.24182 (Δ=2.26e-01)
WARNING    | Alice no-signaling violated at index [2, 1]: 0.17882 vs 0.32113 (Δ=1.42e-01)
WARNING    | Alice no-signaling violated at index [3, 1]: 0.77048 vs 0.94646 (Δ=1.76e-01)
WARNING    | Alice no-signaling violated at index [4, 1]: 0.17083 vs 0.35752 (Δ=1.87e-01)
WARNING    | Alice no-signaling violated at index [5, 1]:

WARNING    |   Bob no-signaling violated at index [0, 1]: 0.75166 vs 0.66706 (Δ=8.46e-02)
WARNING    |   Bob no-signaling violated at index [1, 1]: 0.64843 vs 0.72714 (Δ=7.87e-02)
WARNING    |   Bob no-signaling violated at index [2, 1]: 0.43348 vs 0.90495 (Δ=4.71e-01)
WARNING    |   Bob no-signaling violated at index [3, 1]: 0.62475 vs 0.60236 (Δ=2.24e-02)
WARNING    |   Bob no-signaling violated at index [4, 1]: 0.24834 vs 0.33294 (Δ=8.46e-02)
WARNING    |   Bob no-signaling violated at index [5, 1]: 0.35157 vs 0.27286 (Δ=7.87e-02)
WARNING    |   Bob no-signaling violated at index [6, 1]: 0.56652 vs 0.09505 (Δ=4.71e-01)
WARNING    |   Bob no-signaling violated at index [7, 1]: 0.37525 vs 0.39764 (Δ=2.24e-02)
Sample behavior is no-signaling:  False


Sampling no-signaling behaviors can't reasonably be achieved with rejection sampling from normalized behaviors though, since as $dim(\mathcal{NS}) < dim(\mathcal{B})$, the usual measure of $\mathcal{NS}$ in $\mathcal{B}$ is null. Getting a no-signaling behavior would thus be very, very lucky.

A consequence of this is that we will need to implement uniform sampling on the $\mathcal{NS}$ polytope to sample no-signaling behaviors directly. This can be achieved if we know the polytope's vertices, which is the case in low-dimensions only. The method consists in partitioning the arbitrary bounded polytope in simplices, which we can sample from by weighing them using their volumes, and then using simplex-specific methods to sample uniformly in the chosen simplex.

It also seems that we are able to uniformly sample from a polytope using MCMC methods, as per [Sun and Chen, 2024](https://arxiv.org/abs/2412.06629), using the ``polytopewalk`` module.

### Testing ``polytopewalk``

In [8]:
import polytopewalk as pw

# Get the equations of the no-signaling set
logger.info("Getting the equations of the no-signaling set")
A, b = routed_no_signaling_equations(delta=2, m=2)

logger.info(f"Equations shape: {A.shape}")

# Rank reduce the matrix
logger.info("Rank reducing the matrix")
U, s, Vh = sp.linalg.svd(A)
rank = np.sum(s > 1e-10)
A_reduced = s[:rank, None] * Vh[:rank, :]  # shape: (rank, n)
b_reduced = (U.T[:rank] @ b).reshape(-1, 1)  # shape: (rank, 1)

# Change to sparse representation
logger.info("Changing to sparse representation")
A_comp = sp.sparse.csc_matrix(A_reduced, dtype=np.float64)
b_comp = b_reduced.astype(np.float64)

# Get the polytopewalk objects
logger.info("Getting the polytopewalk objects")
# Walk
logger.info("Getting the walk object")
walk = pw.sparse.SparseHitAndRun()
# Facial reduction
logger.info("Getting the facial reduction object")
fr = pw.FacialReduction()
fr_output = fr.reduce(A_comp, b_comp, k=A_comp.shape[1], sparse=True)
# logger.info(f"Facial reduction output: {fr_output}")
# logger.info(f"Facial reduction output Q: {fr_output.Q}")
# logger.info(f"Facial reduction output z1: {fr_output.z1}")
# logger.info(f"Facial reduction output A: {fr_output.sparse_A}")
# logger.info(f"Facial reduction output b: {fr_output.sparse_b}")
# # Center
logger.info("Getting the center object")
# z_custom = np.linalg.lstsq(
#     fr_output.Q, completely_mixed_behavior.get_vector() - fr_output.z1, rcond=None
#     )[0]
# class CustomCenter(pw.sparse.SparseCenter):
#     def getInitialPoint(self, A, b, k):
#         return completely_mixed_behavior.get_vector()
# sc = CustomCenter()
sc = pw.sparse.SparseCenter()
logger.info(f"Test center point: {sc.getInitialPoint(fr_output.sparse_A, fr_output.sparse_b, rank)}")

# # Run the MCMC
logger.info("Running the MCMC")
samples_comp = pw.sparseFullWalkRun(
    A=fr_output.sparse_A,
    b=fr_output.sparse_b,
    k=A_comp.shape[1],
    num_sim=100,
    walk=walk,
    fr=fr,
    sc=sc,
    burn=1000,
    )

# Map back to the original space
logger.info("Mapping back to the original space")
if fr_output.Q is not None and fr_output.Q.size > 0:
    logger.info(f"Mapping back to the original space with Q: {fr_output.Q}")
    samples_og = (fr_output.Q @ samples_comp.T).T + fr_output.z1.T
else:
    logger.info("Already in the original space, no mapping needed")
    samples_og = samples_comp  # Already in original space


# Check that the point is in the no-signaling set
logger.info("Checking that the point is in the no-signaling set")
sampled_behavior = Behavior(samples_og[0])
print("Sampled behavior: ", sampled_behavior)
print("Sampled behavior is no-signaling: ", sampled_behavior.is_no_signaling(atol=1e-2))

# Reminder of the no-signaling check procedure
from behaviors import display_ns_test_arrays
display_ns_test_arrays(sum_over_b=False, verbose=True)
display_ns_test_arrays(sum_over_b=True)


INFO       | Getting the equations of the no-signaling set
INFO       | Equations shape: (28, 32)
INFO       | Rank reducing the matrix
INFO       | Changing to sparse representation
INFO       | Getting the polytopewalk objects
INFO       | Getting the walk object
INFO       | Getting the facial reduction object
INFO       | Getting the center object
INFO       | Test center point: [ 5.00000000e-01 -7.27487666e-16  2.50000000e-01  5.00000000e-01
  0.00000000e+00  5.00000000e-01  2.50000000e-01  0.00000000e+00
  0.00000000e+00  5.00000000e-01  2.50000000e-01  0.00000000e+00
  5.00000000e-01  0.00000000e+00  2.50000000e-01  5.00000000e-01
  2.50000000e-01  2.50000000e-01  2.50000000e-01  2.50000000e-01
  2.50000000e-01  2.50000000e-01  2.50000000e-01  2.50000000e-01
  2.50000000e-01  2.50000000e-01  2.50000000e-01  2.50000000e-01
  2.50000000e-01  2.50000000e-01  2.50000000e-01  2.50000000e-01]
INFO       | Running the MCMC
INFO       | Mapping back to the original space
INFO       | Al

In [9]:
# help(pw.sparse.SparseCenter)